In [1]:
import os; os.environ['WORKDIR'] = "/home/choij/workspace/ChargedHiggsAnalysis"
import sys; sys.path.insert(0, os.environ['WORKDIR'])

import numpy as np
import pandas as pd
from ROOT import TFile

from itertools import product
from ctypes import c_double
from pprint import pprint

Welcome to JupyROOT 6.26/06


In [3]:
# Arguments
ERAs = ["2016preVFP", "2016postVFP", "2017", "2018"]
MEASUREs = ["DYJets", "ZGamma"]
histkey = "ZCand/mass"

DataStream  = "DoubleMuon"
Conversion  = ["DYJets", "ZGToLLG"]
VV          = ["WZTo3LNu_mllmin4p0_powheg","ZZTo4L_powheg"]
ttX         = ["ttWToLNu", "ttZToLLNuNu", "ttHToNonbb", "tZq", "tHq"]
Rare        = ["WWW", "WWZ", "WZZ", "ZZZ","WWG", "TTG", "TTTT", "VBF_HToZZTo4L", "GluGluHToZZTo4L"]
MCSamples   = Conversion + VV + ttX + Rare

Systematics = ["Central",
               ["NonpromptUp", "NonpromptDown"],
               ["L1PrefireUp", "L1PrefireDown"],
               ["PileUpCorrUp", "PileUpCorrDown"],
               ["MuonMomentumShiftUp", "MuonMomentumShiftDown"],
               ["JetEnShiftUp", "JetEnShiftDown"],
               ["JetResShiftUp", "JetResShiftDown"],
               ["MuonIDSFUp", "MuonIDSFDown"],
               ["DblMuonTrigSFUp", "DblMuonTrigSFDown"]
               ]

In [27]:
class SummaryWriter():
    def __init__(self, era, measure):
        self.era = era
        self.measure = measure
        self.dataframe = {}
        self.__make_index_col()
        
    def __make_index_col(self):
        self.indexCol = ["Sample", "Central", "Stat"]
        for syst in Systematics[1:]:
            self.indexCol.append(syst[0])
            self.indexCol.append(syst[1])
        self.indexCol.append("Total")
        for index in self.indexCol:
            self.dataframe[index] = []
            
    def __estimate_total_err(self, sample):
        # find the index of the sample
        idx = self.dataframe["Sample"].index(sample)
        central = self.dataframe['Central'][idx]
        total = np.power(self.dataframe['Stat'][idx], 2)
        for syst in Systematics[1:]:
            if np.isnan(self.dataframe[syst[0]][idx]):
                continue
            syst_up = self.dataframe[syst[0]][idx] - central
            syst_down = self.dataframe[syst[1]][idx] - central
            total += np.power(max(abs(syst_up), abs(syst_down)), 2)
        return np.sqrt(total)    
            
    def write_data(self):
        f = TFile.Open(f"{os.environ['WORKDIR']}/triLepRegion/output/ROOT/Skim3Mu__/{self.era}/{DataStream}.root")
        
        # first row: data
        self.dataframe['Sample'].append("data")
        h_data = f.Get(f"3Mu/ZGammaRegion/Central/{self.measure}/{histkey}")
        assert h_data; h_data.SetDirectory(0)
        stat = c_double()
        data = h_data.IntegralAndError(0, h_data.GetNbinsX(), stat)
        self.dataframe['Central'].append(data)
        self.dataframe['Stat'].append(stat.value)
        # Systematics
        for index in self.indexCol[3:-1]:
            self.dataframe[index].append(np.nan)
        self.dataframe['Total'].append(self.__estimate_total_err("data"))
        
        # second row: fake
        self.dataframe['Sample'].append("fake")
        h_fake = f.Get(f"3Mu/ZGammaRegion/Nonprompt/{self.measure}/{histkey}")
        assert h_fake; h_fake.SetDirectory(0)
        h_fake_up = f.Get(f"3Mu/ZGammaRegion/NonpromptUp/{self.measure}/{histkey}")
        assert h_fake_up; h_fake_up.SetDirectory(0)
        h_fake_down = f.Get(f"3Mu/ZGammaRegion/NonpromptDown/{self.measure}/{histkey}")
        assert h_fake_down; h_fake_down.SetDirectory(0)
        stat = c_double()
        fake = h_fake.IntegralAndError(0, h_fake.GetNbinsX(), stat)
        self.dataframe['Central'].append(fake)
        self.dataframe['Stat'].append(stat.value)
        # Systematics
        self.dataframe['NonpromptUp'].append(h_fake_up.Integral())
        self.dataframe['NonpromptDown'].append(h_fake_down.Integral())
        for index in self.indexCol[5:-1]:
            self.dataframe[index].append(np.nan)
        self.dataframe['Total'].append(self.__estimate_total_err("fake"))
        
        f.Close()
    
    def write_mc(self):
        # nth row
        for sample in MCSamples:
            f = TFile.Open(f"{os.environ['WORKDIR']}/triLepRegion/output/ROOT/Skim3Mu__/{self.era}/{sample}.root")
            self.dataframe['Sample'].append(sample)
            h_cent = f.Get(f"3Mu/ZGammaRegion/Central/{self.measure}/{histkey}")
            if not h_cent:
                for index in self.indexCol[1:]:
                    self.dataframe[index].append(0.)
                self.dataframe['NonpromptUp'][-1] = np.nan
                self.dataframe['NonpromptDown'][-1] = np.nan
                f.Close()
                continue
            h_cent.SetDirectory(0)
            
            # fill each row
            stat = c_double()
            cent = h_cent.IntegralAndError(0, h_cent.GetNbinsX(), stat)
            self.dataframe['Central'].append(cent)
            self.dataframe['Stat'].append(stat.value)
            self.dataframe['NonpromptUp'].append(np.nan)
            self.dataframe['NonpromptDown'].append(np.nan)
            for index in self.indexCol[5:-1]:
                h_syst = f.Get(f"3Mu/ZGammaRegion/{index}/{self.measure}/{histkey}")
                assert h_syst; h_syst.SetDirectory(0)
                self.dataframe[index].append(h_syst.Integral())
            self.dataframe['Total'].append(self.__estimate_total_err(sample))
            f.Close()
            
    def measure_scale_factor(self):
        self.dataframe['Sample'].append("Scale Factor")
        
        # Central
        data = self.dataframe['Central'][0]
        fake = self.dataframe['Central'][1]
        if self.measure == "DYJets":
            conv = self.dataframe['Central'][2]
        else: # ZGamma
            conv = self.dataframe['Central'][3]
        pred = sum(self.dataframe["Central"][4:])
        sf = (data - (fake+pred))/conv
        self.dataframe['Central'].append(sf)
        
        # Stat and Total
        for index in ['Stat', 'Total']:
            data_up = self.dataframe['Central'][0] + self.dataframe[index][0]
            fake_up = self.dataframe['Central'][1] + self.dataframe[index][1]
            if self.measure == "DYJets":
                conv_up = self.dataframe['Central'][2] + self.dataframe[index][2]
            else:   # ZGamma
                conv_up = self.dataframe['Central'][3] + self.dataframe[index][3]
            pred_up = sum(self.dataframe['Central'][4:]) + sum(self.dataframe[index][4:])
            sf_up = (data_up - (fake_up+pred_up))/conv_up

            data_down = self.dataframe['Central'][0] - self.dataframe[index][0]
            fake_down = self.dataframe['Central'][1] - self.dataframe[index][1]
            if self.measure == "DYJets":
                conv_down = self.dataframe['Central'][2] - self.dataframe[index][2]
            else:   # ZGamma
                conv_down = self.dataframe['Central'][3] - self.dataframe[index][3]
            pred_down = sum(self.dataframe['Central'][4:]) - sum(self.dataframe[index][4:])
            sf_down = (data_down - (fake_down+pred_down))/conv_down
        
            central = sf
            sf = sf_up if abs(sf_up - central) > abs(sf_down - central) else sf_down
            self.dataframe[index].append(sf - central)
        
        # Fake impact
        for index in self.indexCol[3:-1]:
            data = self.dataframe['Central'][0]
            if self.measure == "DYJets":
                conv_idx = 2
            else: # ZGamma
                conv_idx = 3
            is_prompt = np.isnan(self.dataframe[index][1])
            if is_prompt:
                fake = self.dataframe['Central'][1]
                conv = self.dataframe[index][conv_idx]
                pred = sum(self.dataframe[index][4:])
            else:   # Nonprompt
                fake = self.dataframe[index][1]
                conv = self.dataframe['Central'][conv_idx]
                pred = sum(self.dataframe['Central'][4:])
            sf = (data - (fake+pred))/conv
            self.dataframe[index].append(sf)
        
    def make_dataframe(self):
        df = pd.DataFrame(self.dataframe)
        df.set_index("Sample", inplace=True)
        
        # Make readable
        for row, col in product(df.index, df.columns):
            value = df.loc[row, col]
            # no update
            if np.isnan(value):
                continue
    
            central = float(df.loc[row, 'Central'])
            if central == 0:
                continue
    
            if col == "Central":
                df.loc[row, col] = f"{value:.4f}"
            elif col in ["Stat", "Total"]:
                df.loc[row, col] = f"{value:.4f} ({value/central*100:.3f}%)"
            else:
                df.loc[row, col] = f"{value:.4f} ({(value - central)/central*100:.3f}%)"
        self.df = df

In [28]:
with pd.ExcelWriter("../results/Conversion/ScaleFactors.xlsx") as writer:
    for era, measure in product(ERAs, MEASUREs):
        s_writer = SummaryWriter(era, measure)
        s_writer.write_data()
        s_writer.write_mc()
        s_writer.measure_scale_factor()
        s_writer.make_dataframe()
        s_writer.df.to_csv(f"../results/Conversion/ScaleFactors_{era}_{measure}.csv")
        s_writer.df.to_excel(writer, sheet_name=f"{era}-{measure}")

In [23]:
s_writer = SummaryWriter("2016preVFP", "DYJets")
s_writer.write_data()
s_writer.write_mc()
s_writer.measure_scale_factor()
df = s_writer.to_dataframe()

In [24]:
df

,Central,Stat,NonpromptUp,NonpromptDown,L1PrefireUp,L1PrefireDown,PileUpCorrUp,PileUpCorrDown,MuonMomentumShiftUp,MuonMomentumShiftDown,JetEnShiftUp,JetEnShiftDown,JetResShiftUp,JetResShiftDown,MuonIDSFUp,MuonIDSFDown,DblMuonTrigSFUp,DblMuonTrigSFDown,Total
Sample,,,,,,,,,,,,,,,,,,,
data,269.0000,16.4012 (6.097%),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.4012 (6.097%)
fake,64.7872,6.0412 (9.325%),96.3225 (48.675%),40.5620 (-37.392%),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.1088 (49.560%)
DYJets,179.5387,21.6270 (12.046%),NaN,NaN,177.8478 (-0.942%),181.1307 (0.887%),179.5387 (-0.000%),179.5387 (-0.000%),179.5471 (0.005%),179.5352 (-0.002%),179.5336 (-0.003%),179.5529 (0.008%),179.5304 (-0.005%),179.5606 (0.012%),186.9993 (4.155%),172.2523 (-4.058%),179.6213 (0.046%),179.4548 (-0.047%),22.9403 (12.777%)
ZGToLLG,85.5260,2.9640 (3.466%),NaN,NaN,84.8116 (-0.835%),86.1964 (0.784%),85.5260 (0.000%),85.5260 (0.000%),85.6906 (0.192%),85.7403 (0.251%),85.5178 (-0.010%),85.5333 (0.008%),85.5252 (-0.001%),85.5247 (-0.001%),89.3182 (4.434%),81.8264 (-4.326%),85.5645 (0.045%),85.4870 (-0.046%),4.8707 (5.695%)
WZTo3LNu_mllmin4p0_powheg,13.3971,1.0516 (7.850%),NaN,NaN,13.2878 (-0.816%),13.5006 (0.772%),13.3971 (-0.000%),13.3971 (-0.000%),13.2289 (-1.256%),13.3971 (-0.000%),13.3961 (-0.007%),13.3987 (0.012%),13.3967 (-0.003%),13.3977 (0.005%),13.9406 (4.057%),12.8649 (-3.972%),13.4031 (0.044%),13.3910 (-0.046%),1.2007 (8.962%)
ZZTo4L_powheg,83.2884,0.2023 (0.243%),NaN,NaN,82.6094 (-0.815%),83.9068 (0.743%),83.2884 (-0.000%),83.2884 (-0.000%),83.4155 (0.153%),83.1354 (-0.184%),83.2820 (-0.008%),83.2949 (0.008%),83.2896 (0.001%),83.2908 (0.003%),86.8825 (4.315%),79.7825 (-4.209%),83.3271 (0.046%),83.2491 (-0.047%),3.6666 (4.402%)
ttWToLNu,0.0273,0.0114 (41.813%),NaN,NaN,0.0269 (-1.614%),0.0278 (1.885%),0.0273 (0.142%),0.0273 (0.142%),0.0273 (0.142%),0.0273 (0.142%),0.0273 (0.024%),0.0274 (0.265%),0.0274 (0.230%),0.0273 (0.137%),0.0282 (3.276%),0.0265 (-2.942%),0.0273 (0.165%),0.0273 (0.120%),0.0115 (41.967%)
ttZToLLNuNu,0.0897,0.0165 (18.362%),NaN,NaN,0.0886 (-1.176%),0.0907 (1.109%),0.0897 (-0.017%),0.0897 (-0.017%),0.0897 (-0.017%),0.0914 (1.910%),0.0896 (-0.089%),0.0898 (0.094%),0.0897 (0.000%),0.0897 (0.036%),0.0929 (3.564%),0.0865 (-3.534%),0.0897 (0.029%),0.0896 (-0.063%),0.0169 (18.843%)
ttHToNonbb,0.0340,0.0085 (24.888%),NaN,NaN,0.0336 (-1.066%),0.0343 (0.773%),0.0340 (-0.121%),0.0340 (-0.121%),0.0316 (-7.023%),0.0340 (-0.121%),0.0339 (-0.188%),0.0340 (0.008%),0.0340 (-0.083%),0.0340 (-0.122%),0.0354 (4.024%),0.0326 (-4.188%),0.0340 (-0.076%),0.0339 (-0.166%),0.0089 (26.175%)
